# Module 06 — Datasets, DataLoaders & Your First Full Project

**Prerequisites:** Module 05 (Loss Functions, Optimizers & the Training Loop)
**Time:** ~75 minutes

## Learning Objectives

- Implement a custom `Dataset` and explain the two methods it must define.
- Use `DataLoader` to batch, shuffle, and iterate over a dataset.
- Split data into train/validation/test sets and explain why each split exists.
- Combine everything from Modules 01–06 into a complete, working classification project, including evaluation, `model.eval()`/`model.train()`, and saving/loading the trained model.


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


## Why Not Just Use One Big Tensor?

In Module 05, we trained on the *entire* dataset in a single forward pass every epoch — fine for 200 toy samples, but real datasets can have millions of examples that won't fit in memory (or on a GPU) all at once. The standard solution is **mini-batch training**: process a manageable chunk ("batch") of examples at a time, update the model, then move to the next chunk.

PyTorch splits this responsibility into two objects:

- **`Dataset`** — knows how to fetch a *single* example, given its index. Doesn't know anything about batching.
- **`DataLoader`** — wraps a `Dataset` and handles batching, shuffling, and (optionally) parallel loading. Doesn't know anything about *what* the data is — only how to group it.


## Building a Custom `Dataset`

Every `Dataset` subclass must implement exactly two methods:

- **`__len__(self)`** — returns the total number of examples. `DataLoader` uses this to know how many batches exist.
- **`__getitem__(self, index)`** — returns the single example (typically an `(input, target)` pair) at that index.


In [2]:
class ToyDataset(Dataset):
    def __init__(self, features, labels):
        # __init__ is where you'd normally load data from disk, apply preprocessing, etc.
        # Here we just store already-created tensors.
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        # Returns ONE example -- DataLoader is responsible for grouping many of these into a batch.
        return self.features[index], self.labels[index]


torch.manual_seed(0)
features = torch.randn(500, 2)
labels = (features[:, 0] + features[:, 1] > 0).float()   # same rule as Module 05's challenge exercise

dataset = ToyDataset(features, labels)
print("dataset length:", len(dataset))
print("first item:", dataset[0])


dataset length: 500
first item: (tensor([-1.1258, -1.1524]), tensor(0.))


## Wrapping It With `DataLoader`

```python
DataLoader(dataset, batch_size, shuffle=True)
```

`shuffle=True` is used for **training** (so the model doesn't see examples in the same order every epoch, which can bias learning); `shuffle=False` is used for **validation/testing** (order doesn't matter there, and keeping it fixed makes debugging easier).


In [3]:
loader = DataLoader(dataset, batch_size=16, shuffle=True)

# DataLoader is an iterable -- each iteration yields one batch
for batch_features, batch_labels in loader:
    print("batch_features shape:", batch_features.shape)   # (16, 2)
    print("batch_labels shape:  ", batch_labels.shape)      # (16,)
    break   # just look at the first batch

print("\nnumber of batches per epoch:", len(loader))   # 500 / 16, rounded up


batch_features shape: torch.Size([16, 2])
batch_labels shape:   torch.Size([16])

number of batches per epoch: 32


```text
Dataset
  __getitem__(0) -> (feature_0, label_0)
  __getitem__(1) -> (feature_1, label_1)
  __getitem__(2) -> (feature_2, label_2)
       ...                                DataLoader groups batch_size of these
                                           into one batch, and shuffles the order
                                           each epoch if shuffle=True
  __getitem__(N) -> (feature_N, label_N)
```


## Train / Validation / Test Splits

Before building the full project, one more essential concept: splitting your data into three parts.

- **Train** — the data the model actually learns from (its parameters are updated using this).
- **Validation** — held-out data used *during* development to check how well the model generalizes, tune hyperparameters (like learning rate), and decide when to stop training. The model never learns from this directly.
- **Test** — data touched *only once*, at the very end, to report a final, honest performance number. If you tune anything based on test performance, it stops being a trustworthy estimate of real-world performance.

A common split ratio for a small dataset is roughly 70% train / 15% validation / 15% test — exact numbers vary by project size and convention.


In [4]:
from torch.utils.data import random_split

n_total = len(dataset)
n_train = int(0.7 * n_total)
n_val   = int(0.15 * n_total)
n_test  = n_total - n_train - n_val

train_set, val_set, test_set = random_split(
    dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(0),   # fixed seed -> reproducible split
)

print(f"train: {len(train_set)}  val: {len(val_set)}  test: {len(test_set)}")

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=32, shuffle=False)


train: 350  val: 75  test: 75


## `model.train()` vs. `model.eval()`

Some layers (dropout, batch normalization — introduced in later modules) behave *differently* during training versus evaluation. `model.train()` and `model.eval()` tell the model which mode it's in. Our simple model doesn't have those layers yet, but calling these is standard practice on every model, every time, because it costs nothing and prevents subtle bugs the moment such a layer is added later.

Combined with `torch.no_grad()` (Module 03) during evaluation, the full pattern is:

```python
model.train()
# ... training loop ...

model.eval()
with torch.no_grad():
    # ... evaluation loop ...
```


## The Complete Project: Binary Classification Pipeline

Let's assemble everything from Modules 01–06 into one coherent, complete project.


In [5]:
class BinaryClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),   # single logit output
        )

    def forward(self, x):
        return self.net(x)


def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0.0
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        logits = model(batch_x).squeeze(1)     # (batch, 1) -> (batch,) to match batch_y's shape
        loss = loss_fn(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch_y)
    return total_loss / len(loader.dataset)


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch_x, batch_y in loader:
            logits = model(batch_x).squeeze(1)
            loss = loss_fn(logits, batch_y)
            total_loss += loss.item() * len(batch_y)

            predictions = (logits > 0).float()   # logit > 0  <=>  sigmoid(logit) > 0.5
            correct += (predictions == batch_y).sum().item()
            total += len(batch_y)
    return total_loss / total, correct / total


Notice `.squeeze(1)` on the model's output: our model outputs shape `(batch, 1)` (one logit per sample, kept as a column, matching `nn.Linear`'s natural output shape), but `batch_y` from our `Dataset` has shape `(batch,)`. Squeezing makes the shapes match so `nn.BCEWithLogitsLoss` and the accuracy comparison both work correctly — a shape mismatch here wouldn't always error loudly (broadcasting might silently "fix" it into something incorrect), so it's worth tracking explicitly rather than guessing.


In [6]:
torch.manual_seed(0)

model = BinaryClassifier(input_dim=2, hidden_dim=16)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

n_epochs = 30
for epoch in range(n_epochs):
    train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, loss_fn)
    if epoch % 5 == 0 or epoch == n_epochs - 1:
        print(f"epoch {epoch:2d} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} | val_acc {val_acc:.3f}")

test_loss, test_acc = evaluate(model, test_loader, loss_fn)
print(f"\nFinal test accuracy: {test_acc:.3f}")


epoch  0 | train_loss 0.6311 | val_loss 0.5434 | val_acc 0.947
epoch  5 | train_loss 0.1286 | val_loss 0.1198 | val_acc 0.987
epoch 10 | train_loss 0.0583 | val_loss 0.0688 | val_acc 1.000
epoch 15 | train_loss 0.0414 | val_loss 0.0558 | val_acc 1.000
epoch 20 | train_loss 0.0339 | val_loss 0.0491 | val_acc 0.987


epoch 25 | train_loss 0.0284 | val_loss 0.0436 | val_acc 1.000
epoch 29 | train_loss 0.0253 | val_loss 0.0431 | val_acc 1.000

Final test accuracy: 1.000


## Saving and Loading the Trained Model

Save the **`state_dict`** — a dictionary mapping parameter names to their tensor values — rather than the whole model object. `state_dict` is portable and doesn't break if you later refactor your code; saving the raw model object (via plain `pickle`) can break in exactly that situation.


In [7]:
# Save
torch.save(model.state_dict(), "/tmp/binary_classifier.pt")
print("Saved.")

# Load into a freshly constructed model with the SAME architecture
loaded_model = BinaryClassifier(input_dim=2, hidden_dim=16)
loaded_model.load_state_dict(torch.load("/tmp/binary_classifier.pt", weights_only=True))
loaded_model.eval()

# Verify it produces identical predictions to the original
with torch.no_grad():
    original_out = model(features[:5])
    loaded_out = loaded_model(features[:5])
print("Outputs match:", torch.allclose(original_out, loaded_out))


Saved.
Outputs match: True


## 🐛 Debugging Challenge

The evaluation function below has a subtle bug that inflates memory usage and can cause misleading results if `.eval()`-sensitive layers were present. Find it.


In [8]:
def buggy_evaluate(model, loader, loss_fn):
    # model.eval()   <-- MISSING
    total_loss, correct, total = 0.0, 0, 0
    # with torch.no_grad():   <-- MISSING
    for batch_x, batch_y in loader:
        logits = model(batch_x).squeeze(1)
        loss = loss_fn(logits, batch_y)
        total_loss += loss.item() * len(batch_y)
        predictions = (logits > 0).float()
        correct += (predictions == batch_y).sum().item()
        total += len(batch_y)
    return total_loss / total, correct / total

# This still "works" and gives a similar accuracy number here, because our model
# has no dropout/batchnorm layers -- but it silently builds an unused computational
# graph for every batch (wasting memory), and would give WRONG results the moment
# a dropout or batchnorm layer is added to the model.
loss, acc = buggy_evaluate(model, val_loader, loss_fn)
print(loss, acc)


0.04308736238007744 1.0


**Diagnosis:** missing `model.eval()` and `torch.no_grad()`. With this simple model, results look fine — but this is exactly the kind of bug that stays invisible until you add dropout or batch normalization (Module 08+), at which point it silently produces incorrect validation numbers. Always include both, on every evaluation function, as a standing habit.


## Exercises

🟢 **Beginner:** Print the total number of batches produced by `train_loader`, `val_loader`, and `test_loader`. Verify the math: does `len(loader)` roughly equal `len(loader.dataset) / batch_size`, rounded up?

🟡 **Intermediate:** Modify `ToyDataset` to add a simple form of preprocessing inside `__getitem__` — for example, multiply the features by 2 before returning them. Re-run the pipeline and observe whether/how results change (they shouldn't change *dramatically*, since it's just a rescaling, but the loss curve's absolute numbers may shift).

🔴 **Challenge:** Extend the project to track training loss *and* validation loss across all epochs in two Python lists, then plot both on the same chart using `matplotlib` (`import matplotlib.pyplot as plt`). Two curves that both decrease and stay close together indicate healthy learning; validation loss rising while training loss keeps falling is the classic signature of **overfitting** — a topic covered in depth in a later module, but worth recognizing visually now.


In [9]:
# Space for your exercise solutions



## Common Mistakes

- **Forgetting `model.eval()` / `torch.no_grad()`** during validation or testing — invisible with simple models, dangerous once dropout/batchnorm are added.
- **Shuffling the validation/test loader** — unnecessary, and makes debugging harder since results aren't reproducible in the same order run-to-run.
- **Shape mismatches between model output and target** — e.g., `(batch, 1)` vs. `(batch,)` — that silently broadcast into wrong results instead of erroring.
- **Tuning based on test-set performance** — defeats the purpose of having a held-out test set; only look at it once, at the end.
- **Saving the whole model object instead of `state_dict`** — fragile across code refactors.

## Mental Model

Think of `Dataset` as a librarian who can hand you any *one* book by its shelf number (`__getitem__`), and knows the total number of books (`__len__`). `DataLoader` is an assistant who repeatedly asks the librarian for a stack of `batch_size` books at a time, shuffling the shelf order first if asked. Neither one cares what's actually written inside the books — that's the model's job.

## Key Takeaways

- `Dataset` defines how to fetch one example; `DataLoader` handles batching and shuffling on top of it.
- Train/validation/test splits serve different purposes: learning, tuning/monitoring, and final honest evaluation, respectively.
- Always pair `model.train()`/`model.eval()` with the corresponding training/evaluation code, and always wrap evaluation in `torch.no_grad()`.
- Save `state_dict()`, not the raw model object, for portability.

## What's Next

This completes the beginner arc of the course: tensors → autograd → models → training loop → data pipeline → full project. From here, subsequent modules (in the fuller curriculum map) build on this exact foundation to cover convolutional neural networks and image data, regularization techniques (dropout, batch normalization) to fight overfitting, GPU training at scale, and finally restructuring notebook code into a proper, real-world PyTorch project layout (`src/models.py`, `src/train.py`, etc.).

## Checklist

- [ ] I can implement a custom `Dataset` with `__len__` and `__getitem__`
- [ ] I can wrap a `Dataset` in a `DataLoader` and explain what `shuffle` does and when to use it
- [ ] I can explain the purpose of train/validation/test splits
- [ ] I built and trained a complete classification pipeline, including evaluation and saving/loading the model
